# Sub-sampling configuration: interactive explorationTable 4 is produced by `experiments/run_subsampling_sensitivity.py`, whichsweeps the full grid over replicates, checkpoints, and emits the LaTeXfragment. Use that for the reported results.This notebook is for looking at one configuration at a time: how the losstrajectory, runtime and metrics respond to a particular choice of `s` and `b`,without waiting for the whole grid.Because Theorem A.2 makes the sub-sampled objective unbiased for the full lossat every `s`, the parameter governs the variance of the gradient estimaterather than its bias. What the sweep looks for is the point beyond which extrapairs no longer move the estimate, since per-epoch cost grows as O(n' s).

In [ ]:
import os, syssys.path.insert(0, os.path.abspath(".."))   # repository root, so `rnn_agt` importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport rnn_agtfrom rnn_agt import data as Dfrom rnn_agt.seeds import make_seedsfrom rnn_agt.train import TrainConfig, train_model, predictfrom rnn_agt.metrics import evaluateprint("rnn_agt", rnn_agt.__version__)

In [ ]:
import timeS, B = 10, 64          # the configuration to inspectN_TRAIN, CENSORING = 1000, 0.50CHECKPOINTS = (5, 10)seeds = make_seeds(2026)rng = seeds.data()tau = D.calibrate_tau(N_TRAIN, D.f_interaction, "normal", rng, CENSORING,                      D.DEPENDENCE_SPECS["ar1"])tr = D.make_dataset(N_TRAIN, "interaction", "normal", rng,                    dependence="ar1", tau=tau)te = D.make_dataset(2000, "interaction", "normal", rng,                    dependence="ar1", tau=tau)cfg = TrainConfig(model="rnn_agt", epochs=max(CHECKPOINTS), pair_sample_s=S,                  pair_batch_b=B, hidden_dim=64, gru_layers=2, lr=3e-4,                  eval_at_epochs=CHECKPOINTS)t0 = time.time()res = train_model(tr, te, 3, cfg, make_seeds(11))print(f"s={S}, b={B}: {time.time() - t0:.1f}s")for ep, m in sorted(res.checkpoints.items()):    print(f"  epoch {ep:2d}  C={m['test_cindex']:.3f}  AMSE={m['test_amse']:.2f}  "          f"shift={m['location_shift']:+.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.6))ax.plot(range(1, len(res.epoch_losses) + 1), res.epoch_losses, marker="o", ms=3)ax.set_xlabel("epoch"); ax.set_ylabel("mini-batch Gehan-WRS loss")ax.set_title(f"s={S}, b={B}"); ax.grid(alpha=.3)fig.tight_layout(); plt.show()